In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

analysis = pd.read_csv(
    "data/cleaned/analysis_ready_flights.csv"
)

print("Dataset shape:", analysis.shape)
print("Missing arrival delays:", analysis["arr_delay"].isna().sum())

display(analysis.head())

Dataset shape: (327346, 34)
Missing arrival delays: 0


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,season,scheduled_hour,year_weather,month_weather,day_weather,hour_weather,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 10:00:00+00:00,Winter,5,2013.0,1.0,1.0,5.0,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 10:00:00+00:00,Winter,5,2013.0,1.0,1.0,5.0,39.92,24.98,54.81,250.0,14.96014,21.86482,0.0,1011.4,10.0
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 10:00:00+00:00,Winter,5,2013.0,1.0,1.0,5.0,39.02,26.96,61.63,260.0,14.96014,NaN,0.0,1012.1,10.0
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 10:00:00+00:00,Winter,5,2013.0,1.0,1.0,5.0,39.02,26.96,61.63,260.0,14.96014,NaN,0.0,1012.1,10.0
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 11:00:00+00:00,Winter,6,2013.0,1.0,1.0,6.0,39.92,24.98,54.81,260.0,16.11092,23.01560,0.0,1011.7,10.0


In [2]:
overall_stats = analysis["arr_delay"].describe()

print("Overall Arrival Delay Statistics (minutes)")
display(overall_stats.round(2))

print(
    "Percentage of flights arriving late:",
    round((analysis["arr_delay"] > 0).mean() * 100, 2),
    "%"
)

print(
    "Percentage of flights delayed over 15 minutes:",
    round((analysis["arr_delay"] > 15).mean() * 100, 2),
    "%"
)

Overall Arrival Delay Statistics (minutes)


count    327346.00
mean          6.90
std          44.63
min         -86.00
25%         -17.00
50%          -5.00
75%          14.00
max        1272.00
Name: arr_delay, dtype: float64

Percentage of flights arriving late: 40.63 %
Percentage of flights delayed over 15 minutes: 23.71 %


In [3]:
season_order = ["Winter", "Spring", "Summer", "Fall"]

seasonal_stats = (
    analysis.groupby("season")["arr_delay"]
    .agg(
        Flights="count",
        Mean="mean",
        Median="median",
        Std="std",
        Min="min",
        Max="max"
    )
    .reindex(season_order)
    .round(2)
)

print("Arrival Delay by Season (minutes)")
display(seasonal_stats)

Arrival Delay by Season (minutes)


,Flights,Mean,Median,Std,Min,Max
season,,,,,,
Winter,77029,9.04,-2.0,42.47,-70.0,1272.0
Spring,83594,6.81,-5.0,45.41,-86.0,931.0
Summer,84124,12.99,-3.0,52.50,-68.0,1127.0
Fall,82599,-1.22,-9.0,34.79,-68.0,1007.0


In [4]:
airport_stats = (
    analysis.groupby("origin")["arr_delay"]
    .agg(
        Flights="count",
        Mean="mean",
        Median="median",
        Std="std"
    )
    .round(2)
)

print("Arrival Delay by NYC Airport (minutes)")
display(airport_stats)

Arrival Delay by NYC Airport (minutes)


,Flights,Mean,Median,Std
origin,,,,
EWR,117127,9.11,-4.0,45.53
JFK,109079,5.55,-6.0,44.28
LGA,101140,5.78,-5.0,43.86


In [5]:
airline_stats = (
    analysis.groupby("carrier")["arr_delay"]
    .agg(
        Flights="count",
        Mean="mean",
        Median="median",
        Std="std"
    )
    .sort_values("Mean", ascending=False)
    .round(2)
)

print("Arrival Delay by Airline (minutes)")
display(airline_stats)

Arrival Delay by Airline (minutes)


,Flights,Mean,Median,Std
carrier,,,,
F9,681,21.92,6.0,61.65
FL,3175,20.12,5.0,54.09
EV,51108,15.80,-1.0,49.86
YV,544,15.56,-2.0,52.92
OO,29,11.93,-7.0,48.58
MQ,25037,10.77,-1.0,43.17
WN,12044,9.65,-3.0,46.88
B6,54049,9.46,-3.0,42.84
9E,17294,7.38,-7.0,50.09


In [6]:
hourly_stats = (
    analysis.groupby("scheduled_hour")["arr_delay"]
    .agg(
        Flights="count",
        Mean="mean",
        Median="median"
    )
    .sort_index()
    .round(2)
)

print("Arrival Delay by Scheduled Departure Hour")
display(hourly_stats)

Arrival Delay by Scheduled Departure Hour


,Flights,Mean,Median
scheduled_hour,,,
5,1940,-4.80,-8.0
6,25447,-3.38,-8.0
7,22475,-5.30,-10.0
8,26734,-1.11,-8.0
9,19931,-1.45,-8.0
10,16370,0.95,-7.0
11,15689,1.48,-7.0
12,17744,3.49,-5.0
13,19457,6.54,-4.0


In [7]:
numeric_vars = [
    "distance",
    "temp",
    "dewp",
    "humid",
    "wind_speed",
    "precip",
    "visib"
]

numeric_stats = (
    analysis[numeric_vars]
    .describe()
    .T
    .round(2)
)

print("Flight Distance and Weather Summary Statistics")
display(numeric_stats)

Flight Distance and Weather Summary Statistics


,count,mean,std,min,25%,50%,75%,max
distance,327346.0,1048.37,735.91,80.00,509.00,888.00,1389.00,4983.00
temp,325802.0,57.01,17.90,10.94,42.08,57.20,71.96,100.04
dewp,325802.0,41.50,19.33,-9.94,26.06,42.80,57.92,78.08
humid,325802.0,59.21,19.57,12.74,43.74,57.22,74.67,100.00
wind_speed,325741.0,11.06,5.53,0.00,6.90,10.36,14.96,42.58
precip,325819.0,0.00,0.03,0.00,0.00,0.00,0.00,1.21
visib,325819.0,9.29,1.98,0.00,10.00,10.00,10.00,10.00


In [8]:
output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

overall_stats.to_csv(output_dir / "overall_stats.csv")
seasonal_stats.to_csv(output_dir / "seasonal_stats.csv")
airport_stats.to_csv(output_dir / "airport_stats.csv")
airline_stats.to_csv(output_dir / "airline_stats.csv")
hourly_stats.to_csv(output_dir / "hourly_stats.csv")
numeric_stats.to_csv(output_dir / "numeric_stats.csv")

print("All summary statistics tables saved successfully!")

All summary statistics tables saved successfully!
